# 04 Model Experimentation

Comparacion formal de modelos sobre los splits temporales ya materializados en Snowflake, usando el mismo pipeline productivo que luego alimenta la API. Este notebook pertenece a la **segunda etapa**, despues de correr `transform` y construir `STAGING/OBT/ML`.

## Que demuestra este notebook

- que los splits temporales tienen datos reales y no estan vacios
- que el entrenamiento puede ejecutarse con la configuracion operativa de la base de 6 meses
- que el proyecto compara baselines, boostings obligatorios y ensambles bajo un mismo contrato de evaluacion
- que la seleccion del modelo se hace por `validation` y no por `test`

## Decision metodologica

El pipeline usa una estrategia hibrida: entrenamiento incremental real para algoritmos que lo soportan y muestra controlada para modelos que no tienen `partial_fit`. La evaluacion se hace siempre por lotes para respetar la restriccion de memoria del proyecto.

Cada experimento debe dejar auditoria explicita de features usadas, version del contrato y si se activo o no el enriquecimiento geografico opcional. La seleccion del modelo se resuelve unicamente con `validation`; `test` queda reservado para verificacion final y no para tuning.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = /home/pabseb/DataMining/final-project/price-prediction-ml-end-to-end


In [2]:
import pandas as pd

from src.data.ingestion import fetch_sample
from src.models.train_model import train_out_of_core
from src.utils.config import get_settings

pd.set_option('display.max_columns', 100)
settings = get_settings()
train_preview = fetch_sample(f"SELECT * FROM {settings.train_table}", limit=2000, settings=settings)
val_preview = fetch_sample(f"SELECT * FROM {settings.val_table}", limit=2000, settings=settings)
test_preview = fetch_sample(f"SELECT * FROM {settings.test_table}", limit=2000, settings=settings)
assert not train_preview.empty and not val_preview.empty and not test_preview.empty, 'Uno de los splits esta vacio.'
pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows_preview': [len(train_preview), len(val_preview), len(test_preview)],
    'min_pickup': [train_preview['pickup_datetime'].min(), val_preview['pickup_datetime'].min(), test_preview['pickup_datetime'].min()],
    'max_pickup': [train_preview['pickup_datetime'].max(), val_preview['pickup_datetime'].max(), test_preview['pickup_datetime'].max()],
})


,split,rows_preview,min_pickup,max_pickup
0,train,2000,2025-01-14 17:18:04,2025-01-14 21:01:19
1,validation,2000,2025-01-26 16:42:20,2025-01-26 20:59:44
2,test,2000,2025-01-28 00:00:24,2025-01-28 09:59:56


In [3]:
preview_target = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'fare_mean': [train_preview['fare_amount'].mean(), val_preview['fare_amount'].mean(), test_preview['fare_amount'].mean()],
    'fare_median': [train_preview['fare_amount'].median(), val_preview['fare_amount'].median(), test_preview['fare_amount'].median()],
})
preview_target


,split,fare_mean,fare_median
0,train,14.66385,11.4
1,validation,18.62760,12.1
2,test,18.19075,12.1


In [4]:
results = train_out_of_core(settings=settings)
results


{'selected_model': 'hist_gradient_boosting',
 'artifact_path': 'data/models/nyc_taxi_fare_baseline.joblib',
 'dummy_metrics': {'val_rmse': 16.172371915006135,
  'test_rmse': 15.784848160661298},
 'incremental_metrics': {'val_rmse': 20.14687657393106,
  'test_rmse': 153.27401686200128},
 'hist_gradient_boosting_metrics': {'val_rmse': 5.508731628599747,
  'test_rmse': 5.300408188426628}}

In [5]:
comparison = pd.DataFrame(results['model_results'])[
    ['model', 'training_strategy', 'rubric_status', 'required', 'sample_rows', 'val_rmse', 'test_rmse', 'feature_contract_version', 'zone_lookup_enabled']
].sort_values('val_rmse')
comparison


,model,val_rmse,test_rmse
2,HistGradientBoostingRegressor,5.508732,5.300408
0,DummyRegressor,16.172372,15.784848
1,SGDRegressor,20.146877,153.274017


In [ ]:
pd.DataFrame({
    'feature': results['feature_audit']['model_feature_columns'],
    'feature_set': 'model_input',
})


In [6]:
print('Selected model:', results['selected_model'])
print('Artifact path:', results['artifact_path'])


Selected model: hist_gradient_boosting
Artifact path: data/models/nyc_taxi_fare_baseline.joblib


## Conclusiones e interpretacion

- Antes de comparar RMSE, debe verificarse que `unavailable_required_models` este vacio.
- La eleccion del modelo final se hace con `val_rmse`; `test_rmse` se reserva para validacion final.
- Si un modelo mas complejo mejora poco frente al baseline fuerte, el equipo debe justificar si el costo computacional se sostiene.
- Cuando esta ventana de 6 meses quede estable, el siguiente paso correcto es extender exactamente el mismo flujo al periodo historico completo sin cambiar la logica del pipeline.
